In [1]:
# Cell 1: Import Libraries and Setup
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Constants for the analysis
PREDICTION_CATEGORIES = {
    0: 'No prediction',
    1: 'High availability (>80%)',
    2: 'Medium availability (30-80%)',
    3: 'Low availability (<10%)'
}

# Cell 2: Load and Process Data
def load_and_process_data(file_path):
    """Load and process the parking data."""
    try:
        # Load the data
        df = pd.read_csv(file_path)
        
        # Convert datetime
        df['datetime'] = pd.to_datetime(df['datetime'])
        
        # Add time-based features
        df['hour'] = df['datetime'].dt.hour
        df['minute'] = df['datetime'].dt.minute
        df['day_of_week'] = df['datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
        df['is_weekend'] = df['day_of_week'].isin([5, 6])
        
        # Create time slot index (each 5 minutes)
        df['time_slot'] = df['hour'] * 12 + (df['minute'] // 5)
        
        print("Data Processing Summary:")
        print("-" * 50)
        print(f"Total records: {len(df):,}")
        print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")
        print("\nParking Zone Types:")
        print(df['tipo'].value_counts())
        print("\nOccupancy Level Distribution:")
        print(df['occupancy_level'].value_counts().sort_index())
        
        return df
    
    except Exception as e:
        print(f"Error in data loading: {str(e)}")
        return None

# Test the data loading
df = load_and_process_data('/Users/adrianiraeguialvear/OnSpot_Predictive_Model/data/feature_engineered_data.csv')

# Cell 3: Create a simple test plot
if df is not None:
    try:
        # Create a simple line plot of average occupancy by hour
        fig = px.line(
            df.groupby('hour')['occupancy_level'].mean().reset_index(),
            x='hour',
            y='occupancy_level',
            title='Average Occupancy by Hour'
        )
        fig.show()
    except Exception as e:
        print(f"Error creating test plot: {str(e)}")

Data Processing Summary:
--------------------------------------------------
Total records: 1,596,000
Date range: 2025-01-31 07:00:00+00:00 to 2025-02-01 06:55:00+00:00

Parking Zone Types:
tipo
VERDA    1365024
BLAVA     230976
Name: count, dtype: int64

Occupancy Level Distribution:
occupancy_level
0    1578144
1       1044
2        755
3      16057
Name: count, dtype: int64


In [6]:
def create_blava_analysis(df):
    """Create analysis plots for BLAVA parking zones"""
    try:
        # Filter for BLAVA zones only
        blava_df = df[df['tipo'] == 'BLAVA'].copy()
        
        print("BLAVA Analysis Summary:")
        print("-" * 50)
        print(f"Total BLAVA records: {len(blava_df):,}")
        print("\nOccupancy Level Distribution:")
        print(blava_df['occupancy_level'].value_counts().sort_index())
        
        # Create subplots
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Average Occupancy by Hour of Day',
                'Average Occupancy by Day of Week',
                'Occupancy Level Distribution',
                'Time Slot Analysis'
            )
        )

        # 1. Hourly pattern with error bars
        hourly_stats = blava_df.groupby('hour')['occupancy_level'].agg(['mean', 'std', 'count']).reset_index()
        fig.add_trace(
            go.Scatter(
                x=hourly_stats['hour'],
                y=hourly_stats['mean'],
                mode='lines+markers',
                name='Hourly Average',
                error_y=dict(
                    type='data',
                    array=hourly_stats['std'],
                    visible=True
                ),
                text=[f'n={x:,}' for x in hourly_stats['count']],
                hovertemplate='Hour: %{x}<br>Avg Occupancy: %{y:.2f}<br>%{text}'
            ),
            row=1, col=1
        )

        # 2. Daily pattern with error bars
        daily_stats = blava_df.groupby('day_of_week')['occupancy_level'].agg(['mean', 'std', 'count']).reset_index()
        fig.add_trace(
            go.Bar(
                x=daily_stats['day_of_week'],
                y=daily_stats['mean'],
                error_y=dict(
                    type='data',
                    array=daily_stats['std']
                ),
                text=[f'n={x:,}' for x in daily_stats['count']],
                textposition='auto',
                name='Daily Average'
            ),
            row=1, col=2
        )

        # 3. Occupancy level distribution
        level_counts = blava_df['occupancy_level'].value_counts().sort_index()
        fig.add_trace(
            go.Bar(
                x=[PREDICTION_CATEGORIES[i] for i in level_counts.index],
                y=level_counts.values,
                text=[f'n={x:,}' for x in level_counts.values],
                textposition='auto',
                name='Occupancy Levels'
            ),
            row=2, col=1
        )

        # 4. Time slot analysis with rolling average
        time_slot_stats = blava_df.groupby('time_slot')['occupancy_level'].agg(['mean', 'count']).reset_index()
        fig.add_trace(
            go.Scatter(
                x=time_slot_stats['time_slot'],
                y=pd.Series(time_slot_stats['mean']).rolling(window=12, center=True).mean(),
                mode='lines',
                name='Time Slot Average (1h rolling)',
                text=[f'n={x:,}' for x in time_slot_stats['count']],
                hovertemplate='Time Slot: %{x}<br>Avg Occupancy: %{y:.2f}<br>%{text}'
            ),
            row=2, col=2
        )

        # Update layout
        fig.update_layout(
            height=800,
            showlegend=True,
            title_text='BLAVA Parking Zone Analysis',
            template='plotly_white'
        )

        # Update axes labels and formats
        fig.update_xaxes(title_text='Hour of Day', row=1, col=1, dtick=1)
        fig.update_xaxes(
            title_text='Day of Week', 
            row=1, col=2,
            ticktext=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
            tickvals=[0, 1, 2, 3, 4, 5, 6]
        )
        fig.update_xaxes(title_text='Occupancy Category', row=2, col=1)
        fig.update_xaxes(title_text='Time Slot (5-min intervals)', row=2, col=2)

        fig.update_yaxes(title_text='Average Occupancy', row=1, col=1)
        fig.update_yaxes(title_text='Average Occupancy', row=1, col=2)
        fig.update_yaxes(title_text='Count', row=2, col=1)
        fig.update_yaxes(title_text='Average Occupancy', row=2, col=2)

        return fig

    except Exception as e:
        print(f"Error creating BLAVA analysis: {str(e)}")
        return None

# Create and display the BLAVA analysis
blava_fig = create_blava_analysis(df)
if blava_fig is not None:
    blava_fig.show()

BLAVA Analysis Summary:
--------------------------------------------------
Total BLAVA records: 230,976

Occupancy Level Distribution:
occupancy_level
0    213120
1      1044
2       755
3     16057
Name: count, dtype: int64


In [8]:
def analyze_occupancy_patterns(df):
    """Analyze the granularity and patterns of occupancy predictions"""
    try:
        # Filter for BLAVA zones
        blava_df = df[df['tipo'] == 'BLAVA'].copy()
        
        # Create subplots for detailed analysis
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Raw Occupancy Values Distribution',
                'Occupancy Changes Over Time',
                'Value Change Patterns',
                'Value Transitions Heatmap'
            )
        )
        
        # 1. Distribution of raw values
        unique_values = blava_df['occupancy_level'].value_counts().sort_index()
        fig.add_trace(
            go.Bar(
                x=[f"Level {k}" for k in unique_values.index],
                y=unique_values.values,
                text=[f'{v:,} ({v/len(blava_df)*100:.1f}%)' for v in unique_values.values],
                textposition='auto',
                name='Raw Values'
            ),
            row=1, col=1
        )
        
        # 2. Changes over time
        time_series = blava_df.sort_values('datetime').set_index('datetime')
        changes = time_series['occupancy_level'].diff()
        fig.add_trace(
            go.Histogram(
                x=changes,
                name='Value Changes',
                nbinsx=50
            ),
            row=1, col=2
        )
        
        # 3. Value change patterns (bar chart instead of pie)
        time_series['value_change'] = time_series['occupancy_level'].diff().ne(0)
        consecutive_periods = time_series['value_change'].value_counts()
        fig.add_trace(
            go.Bar(
                x=['Changed', 'Unchanged'],
                y=[consecutive_periods.get(True, 0), consecutive_periods.get(False, 0)],
                text=[f'{consecutive_periods.get(True, 0):,}', f'{consecutive_periods.get(False, 0):,}'],
                textposition='auto',
                name='Value Changes'
            ),
            row=2, col=1
        )
        
        # 4. Value transitions
        prev_value = time_series['occupancy_level'].shift()
        transitions = pd.crosstab(prev_value, time_series['occupancy_level'])
        fig.add_trace(
            go.Heatmap(
                z=transitions.values,
                x=transitions.columns,
                y=transitions.index,
                colorscale='Viridis',
                name='Transitions'
            ),
            row=2, col=2
        )
        
        # Update layout
        fig.update_layout(
            height=800,
            title_text='Occupancy Pattern Analysis',
            showlegend=True
        )
        
        return fig
        
    except Exception as e:
        print(f"Error in pattern analysis: {str(e)}")
        return None

# Run the analysis
pattern_fig = analyze_occupancy_patterns(df)
if pattern_fig is not None:
    pattern_fig.show()

# Print analysis of the patterns we see in the data
print("\nDetailed Pattern Analysis:")
print("-" * 50)
print("\n1. Time Coverage Analysis:")
print("- Data is collected in 5-minute intervals")
print("- Zero occupancy during non-operational hours (19:00-06:59)")

print("\n2. Occupancy Level Patterns:")
print("Operational Hours (7:00-18:59):")
blava_df = df[df['tipo'] == 'BLAVA']
operational = blava_df[blava_df['hour'].between(7, 18)]
for level in sorted(operational['occupancy_level'].unique()):
    count = len(operational[operational['occupancy_level'] == level])
    pct = count / len(operational) * 100
    print(f"- Level {level}: {count:,} records ({pct:.1f}%)")

print("\n3. Transition Analysis:")
print("Most common transitions:")
transitions = pd.crosstab(
    blava_df['occupancy_level'].shift(),
    blava_df['occupancy_level']
)
for i in range(4):
    for j in range(4):
        if transitions.iloc[i,j] > 1000:
            print(f"- Level {i} → Level {j}: {transitions.iloc[i,j]:,} times")

print("\n4. Data Quality Concerns:")
print("- Zero values dominate non-operational hours (100%)")
print("- Limited variation in predictions during operational hours")
print("- High self-transition rates suggest potential granularity issues")


Detailed Pattern Analysis:
--------------------------------------------------

1. Time Coverage Analysis:
- Data is collected in 5-minute intervals
- Zero occupancy during non-operational hours (19:00-06:59)

2. Occupancy Level Patterns:
Operational Hours (7:00-18:59):
- Level 0: 97,632 records (84.5%)
- Level 1: 1,044 records (0.9%)
- Level 2: 755 records (0.7%)
- Level 3: 16,057 records (13.9%)

3. Transition Analysis:
Most common transitions:
- Level 0 → Level 0: 198,047 times
- Level 0 → Level 3: 13,551 times
- Level 3 → Level 0: 13,550 times
- Level 3 → Level 3: 2,271 times

4. Data Quality Concerns:
- Zero values dominate non-operational hours (100%)
- Limited variation in predictions during operational hours
- High self-transition rates suggest potential granularity issues


In [9]:
def analyze_non_operational_hours(df):
    """Analyze patterns during non-operational hours to validate zero values"""
    try:
        # Filter for BLAVA zones
        blava_df = df[df['tipo'] == 'BLAVA'].copy()
        
        # Define operational periods
        blava_df['is_operational'] = blava_df['hour'].between(7, 18)
        
        # Create subplots
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Hour Distribution of Zero vs Non-Zero Values',
                'Transitions Around Operational Hours',
                'Value Distribution by Hour',
                'Pattern Around Operating Hours'
            )
        )

        # 1. Hour Distribution of Zero vs Non-Zero Values
        hourly_zeros = pd.crosstab(blava_df['hour'], blava_df['occupancy_level'] == 0)
        fig.add_trace(
            go.Bar(
                name='Non-Zero Values',
                x=hourly_zeros.index,
                y=hourly_zeros[False].values if False in hourly_zeros.columns else [0] * 24,
                marker_color='blue'
            ),
            row=1, col=1
        )
        fig.add_trace(
            go.Bar(
                name='Zero Values',
                x=hourly_zeros.index,
                y=hourly_zeros[True].values,
                marker_color='red'
            ),
            row=1, col=1
        )

        # 2. Transitions around operational hours
        transition_hours = [6, 7, 18, 19]  # Hours around transitions
        transition_data = blava_df[blava_df['hour'].isin(transition_hours)].copy()
        transition_data['minute_of_day'] = transition_data['hour'] * 60 + transition_data['minute']
        
        transition_stats = transition_data.groupby(['minute_of_day', 'occupancy_level']).size().unstack(fill_value=0)
        fig.add_trace(
            go.Scatter(
                x=transition_stats.index,
                y=transition_stats[0],  # Zero values
                name='Level 0',
                mode='lines',
                line=dict(color='red')
            ),
            row=1, col=2
        )
        for level in [1, 2, 3]:
            if level in transition_stats.columns:
                fig.add_trace(
                    go.Scatter(
                        x=transition_stats.index,
                        y=transition_stats[level],
                        name=f'Level {level}',
                        mode='lines'
                    ),
                    row=1, col=2
                )

        # 3. Value Distribution by Hour
        hourly_stats = blava_df.groupby('hour')['occupancy_level'].value_counts().unstack(fill_value=0)
        fig.add_trace(
            go.Heatmap(
                z=hourly_stats.values,
                x=[f'Level {i}' for i in hourly_stats.columns],
                y=hourly_stats.index,
                colorscale='Viridis'
            ),
            row=2, col=1
        )

        # 4. Pattern Around Operating Hours
        edge_hours = blava_df[blava_df['hour'].isin([6, 7, 18, 19])].copy()
        edge_hours['minute_of_day'] = edge_hours['hour'] * 60 + edge_hours['minute']
        avg_by_minute = edge_hours.groupby('minute_of_day')['occupancy_level'].mean()
        
        fig.add_trace(
            go.Scatter(
                x=avg_by_minute.index,
                y=avg_by_minute.values,
                mode='lines+markers',
                name='Average Occupancy'
            ),
            row=2, col=2
        )

        # Update layout
        fig.update_layout(
            height=800,
            title_text='Non-Operational Hours Analysis',
            barmode='stack'
        )

        return fig

    except Exception as e:
        print(f"Error in non-operational analysis: {str(e)}")
        return None

# Run the analysis
non_op_fig = analyze_non_operational_hours(df)
if non_op_fig is not None:
    non_op_fig.show()

# Print detailed analysis
print("\nNon-Operational Hours Analysis:")
print("-" * 50)

blava_df = df[df['tipo'] == 'BLAVA'].copy()
blava_df['is_operational'] = blava_df['hour'].between(7, 18)

# Analyze transitions at boundaries
print("\n1. Transition Analysis:")
for hour in [6, 7, 18, 19]:
    hour_data = blava_df[blava_df['hour'] == hour]
    print(f"\nHour {hour}:00:")
    print(f"Total records: {len(hour_data):,}")
    print("Value distribution:")
    print(hour_data['occupancy_level'].value_counts().sort_index())

# Analyze patterns in zero values
print("\n2. Zero Value Analysis:")
zero_records = blava_df[blava_df['occupancy_level'] == 0]
print(f"Total zero records: {len(zero_records):,}")
print("\nZero values by operational status:")
print(pd.crosstab(blava_df['is_operational'], blava_df['occupancy_level'] == 0))

# Check for any non-zero values during non-operational hours
non_op_nonzero = blava_df[
    (~blava_df['is_operational']) & 
    (blava_df['occupancy_level'] != 0)
]
print("\n3. Non-Zero Values During Non-Operational Hours:")
print(f"Count: {len(non_op_nonzero)}")
if len(non_op_nonzero) > 0:
    print("\nDistribution:")
    print(non_op_nonzero['occupancy_level'].value_counts())


Non-Operational Hours Analysis:
--------------------------------------------------

1. Transition Analysis:

Hour 6:00:
Total records: 9,624
Value distribution:
occupancy_level
0    9624
Name: count, dtype: int64

Hour 7:00:
Total records: 9,624
Value distribution:
occupancy_level
0    8136
1     108
2      96
3    1284
Name: count, dtype: int64

Hour 18:00:
Total records: 9,624
Value distribution:
occupancy_level
0    8136
1      72
2      54
3    1362
Name: count, dtype: int64

Hour 19:00:
Total records: 9,624
Value distribution:
occupancy_level
0    9624
Name: count, dtype: int64

2. Zero Value Analysis:
Total zero records: 213,120

Zero values by operational status:
occupancy_level  False   True 
is_operational                
False                0  115488
True             17856   97632

3. Non-Zero Values During Non-Operational Hours:
Count: 0
